In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import learning_agent
import pandas as pd
from dotenv import load_dotenv
from collections import Counter
from tqdm.auto import tqdm
import json

load_dotenv()

In [ ]:
query = "Extract the numeric part from max_power and calculate statistics"

!truncate -s 0 log.jsonl
!echo "{query}\n\n\n" | uv run analyze-csv --config ../config/insurance-config.yaml &> out.txt
out = !cat out.txt
out = "\n".join(out)

with open("log.jsonl", "r") as f:
    log_content = f.read()

logs = [json.loads(l) for l in log_content.split("\n") if l != ""]

In [ ]:
# print(out)
"error" in out.lower(), Counter([l["level"] for l in logs])

In [ ]:
import json
from learning_agent.logging.event_log_html import render_event_log_html
from IPython.display import display, HTML, Markdown

In [ ]:
with open('log.jsonl') as f:
    events = [json.loads(line) for line in f if line.strip()]

display(HTML(render_event_log_html(events)))

In [ ]:
import json

with open("../insurance_queries.json", "r") as f:
    queries = json.load(f)

In [ ]:
results = {}
for subtitle, subqueries in tqdm(queries["basic_analysis"].items()):
    results[subtitle] = []

    for query in subqueries:
        !truncate -s 0 log.jsonl
        !echo "{query}\n\n\n" | uv run analyze-csv --config ../config/insurance-config.yaml &> out.txt
        print("Error:", "error" in out.lower())
        with open("log.jsonl", "r") as f:
            log_content = f.read()
        logs = [json.loads(l) for l in log_content.split("\n") if l != ""]
        results[subtitle].append(logs)

In [ ]:
for subtitle, subqueries in tqdm(queries["basic_analysis"].items()):
    display(Markdown(f"### {subtitle}"))
    for query, logs in zip(subqueries, results[subtitle]):
        display(Markdown(f"##### {query}"))
        display(HTML(render_event_log_html(logs)))

In [ ]:
results = {}
for subtitle, subqueries in tqdm(queries["failure_mode_queries"].items()):
    results[subtitle] = []

    for query in subqueries:
        !truncate -s 0 log.jsonl
        !echo "{query}\n\n\n" | uv run analyze-csv --config ../config/insurance-config.yaml &> out.txt
        print("Error:", "error" in out.lower())
        with open("log.jsonl", "r") as f:
            log_content = f.read()
        logs = [json.loads(l) for l in log_content.split("\n") if l != ""]
        results[subtitle].append(logs)

In [ ]:
for subtitle, subqueries in tqdm(queries["failure_mode_queries"].items()):
    display(Markdown(f"### {subtitle}"))
    for query, logs in zip(subqueries, results[subtitle]):
        display(Markdown(f"##### {query}"))
        display(HTML(render_event_log_html(logs)))

In [ ]:
print("""
result = pd.DataFrame()\n\n# Step 1: Check if 'subscription_length' column exists\nif 'subscription_length' in df.columns:\n    # Step 2: Convert 'subscription_length' to numeric\n    df['subscription_length'] = pd.to_numeric(df['subscription_length'], errors='coerce')\n    \n    # Step 3: Drop rows with missing values in 'subscription_length'\n    df = df.dropna(subset=['subscription_length'])\n    \n    # Step 4: Define a threshold for subscription_length. \n    # Assuming policies with less than 10 months of subscription length are likely to claim.\n    threshold = 10\n    \n    # Filtering policies likely to have claims based on the threshold.\n    result = df[df['subscription_length'] < threshold]\nelse:\n    # If the condition is not met, return an empty DataFrame\n    result = pd.DataFrame()
""")

In [ ]:
import json
from learning_agent.logging.event_log_analytics import analyze_event_logs
from IPython.display import display

# Load your log file
with open('log.jsonl') as f:
    events = [json.loads(line) for line in f if line.strip()]

# Get all visualizations
viz = analyze_event_logs(events)

# Display them in your notebook
for name, fig in viz.items():
    print(f"\n{'='*50}")
    print(f"�� {name.replace('_', ' ').title()}")
    print(f"{'='*50}")
    display(fig)